In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import numpy as np
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================
BASE_PATH = '..' 
INPUT_DIR = os.path.join(BASE_PATH, 'includes', 'dados')
MAPS_DIR = os.path.join(BASE_PATH, 'includes', 'maps', 'itapua')

# Main Input (The 19k Master Table)
MASTER_INPUT = os.path.join(INPUT_DIR, '..', 'Tabela_consumidores_Itapua.csv')

# Complementary Data
FILE_CONS_2015 = os.path.join(INPUT_DIR, '..', 'Tabela_consumo_medio_Itapua_2015_12m.csv')
FILE_INCOME = os.path.join(BASE_PATH, 'includes', 'ibge_censo2022', 'Agregados_por_setores_renda_responsavel_BR.csv')
FILE_CENSUS = os.path.join(BASE_PATH, 'includes', 'ibge_censo2022', 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv')
SHAPEFILE_PATH = os.path.join(MAPS_DIR, 'Mapa_Itapua.shp')

# Output Path
OUTPUT_FILE = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_com_setor_comportamento_e_renda_2015.csv')

# Constants
CRS_SOURCE = "EPSG:4326"  # WGS84
CRS_TARGET = "EPSG:31984" # SIRGAS 2000 / UTM 24S
PNAD_FACTOR_2015 = 0.835 
THRESHOLD_ENVIRONMENTALIST = 100.0
THRESHOLD_WASTEFUL = 121.5

def create_sector_mapping(df, sector_column='CD_SETOR_ORIGINAL', output_dir='..', mapping_filename='de_para_setores_consumidores_2015.csv'):
    """
    Creates a unique sector mapping to S000 format (S001, S002, ...)
    and saves a CSV file with the DE-PARA mapping for future reference.
    Includes household count per sector.
    """
    
    # Get unique sectors and sort for consistency
    unique_sectors = sorted(df[sector_column].dropna().unique())
    
    # Calculate number of households per original sector
    households_per_sector = df.groupby(sector_column).size().to_dict()
    
    # Create mapping: original -> S001, S002, ...
    sector_mapping = {}
    for i, sector in enumerate(unique_sectors, start=1):
        new_id = f'S{i:03d}'  # Always starts at S001
        sector_mapping[sector] = new_id
    
    # Apply mapping to DataFrame
    df['CD_SETOR_SIMPLIFICADO'] = df[sector_column].map(sector_mapping)
    
    # Create and save DE-PARA file
    de_para_df = pd.DataFrame([
        {
            'SETOR_ORIGINAL': orig, 
            'SETOR_SIMPLIFICADO': simp,
            'NUM_HOUSEHOLDS': households_per_sector.get(orig, 0)
        }
        for orig, simp in sector_mapping.items()
    ])
    de_para_df = de_para_df.sort_values('SETOR_SIMPLIFICADO').reset_index(drop=True)
    de_para_df['INDEX'] = de_para_df.index + 1
    de_para_df['CREATION_DATE'] = pd.Timestamp.now().strftime('%Y-%m-%d')
    
    # Reorder columns for better readability
    de_para_df = de_para_df[['INDEX', 'SETOR_SIMPLIFICADO', 'SETOR_ORIGINAL', 'NUM_HOUSEHOLDS', 'CREATION_DATE']]
    
    # Save CSV
    mapping_path = os.path.join(output_dir, mapping_filename)
    de_para_df.to_csv(mapping_path, index=False, sep=';', encoding='utf-8-sig')
    
    print(f"\n--- Sector Mapping ---")
    print(f"Unique sectors mapped: {len(unique_sectors)}")
    print(f"Format: S001 to S{len(unique_sectors):03d}")
    print(f"DE-PARA file saved at: {mapping_path}")
    print(f"Total households mapped: {de_para_df['NUM_HOUSEHOLDS'].sum()}")
    
    return df, sector_mapping

# ==========================================
# 2. Helper Functions
# ==========================================
def clean_income_value(value):
    if isinstance(value, str):
        if value.strip().upper() == 'X': return np.nan
        value = value.replace('.', '').replace(',', '.')
    try: return float(value)
    except: return np.nan

def classify_profile(daily_liters):
    if daily_liters < THRESHOLD_ENVIRONMENTALIST: return 'AMBIENTALISTA'
    elif daily_liters <= THRESHOLD_WASTEFUL: return 'MODERADO'
    else: return 'PERDULARIO'

def main():
    print("--- Starting Robust Baseline Generation (19k Master) ---")

    # ==========================================
    # 3. Load Master Table and Convert Coordinates
    # ==========================================
    print("Step 1: Loading Master Table and converting to UTM X/Y...")
    df = pd.read_csv(MASTER_INPUT, sep=';')
    original_count = len(df)
    
    # ADJUSTMENT: Do NOT dropna. Fill missing coordinates with 0 to preserve records.
    df['LAT_GEO'] = df['LAT_GEO'].fillna(0)
    df['LONG_GEO'] = df['LONG_GEO'].fillna(0)

    # Coordinate Transformation (Lat/Lon -> UTM 24S)
    transformer = Transformer.from_crs(CRS_SOURCE, CRS_TARGET, always_xy=True)
    x_vals, y_vals = transformer.transform(df["LONG_GEO"].values, df["LAT_GEO"].values)
    
    # ADJUSTMENT: Handle invalid projection values (Inf/NaN) to keep rows alive in GAMA
    df["X"] = np.where(np.isinf(x_vals) | np.isnan(x_vals), 0.0, x_vals)
    df["Y"] = np.where(np.isinf(y_vals) | np.isnan(y_vals), 0.0, y_vals)

    # ==========================================
    # 4. Mapping agents to Census Tracts (Spatial Join)
    # ==========================================
    print("Step 2: Performing Spatial Join with Census Tracts...")
    census_tracts = gpd.read_file(SHAPEFILE_PATH)[['CD_SETOR', 'geometry']]
    geometry = [Point(xy) for xy in zip(df['LONG_GEO'], df['LAT_GEO'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_SOURCE)
    
    gdf = gdf.to_crs(census_tracts.crs)
    
    # ADJUSTMENT: Use Left Join to ensure 100% of agents are kept
    df_spatial = gpd.sjoin(gdf, census_tracts, how="left", predicate="within")
    df_spatial.drop(columns=['geometry', 'index_right'], inplace=True)
    
    # ADJUSTMENT: Fill missing sectors with a placeholder to avoid errors in processing
    df_spatial['CD_SETOR'] = df_spatial['CD_SETOR'].fillna("000000000000000")
    df_spatial['CD_SETOR_ORIGINAL'] = df_spatial['CD_SETOR'].astype(str)

    # ==========================================
    # 5. Integrate IBGE Socio-Economic Data
    # ==========================================
    print("Step 3: Integrating IBGE Census Data...")
    
    # Income per sector
    income_df = pd.read_csv(FILE_INCOME, sep=';')
    income_df['VL_RENDA_MEDIA_RESPONSAVEL'] = income_df['V06004'].apply(clean_income_value)
    income_df['CD_SETOR'] = income_df['CD_SETOR'].astype(str)
    
    # Household characteristics
    census_df = pd.read_csv(FILE_CENSUS, sep=';')
    census_df['VAL_V5'] = pd.to_numeric(census_df['V00005'], errors='coerce')
    census_df['VAL_V1'] = pd.to_numeric(census_df['V00001'], errors='coerce')
    census_df['NN_MEDIA_MORADORES_IBGE'] = census_df['VAL_V5'] / census_df['VAL_V1']
    census_df['CD_SETOR'] = census_df['CD_setor'].astype(str)

    # Merge socio-economic data
    df_enriched = pd.merge(df_spatial, income_df[['CD_SETOR', 'VL_RENDA_MEDIA_RESPONSAVEL']], on='CD_SETOR', how='left')
    df_enriched = pd.merge(df_enriched, census_df[['CD_SETOR', 'NN_MEDIA_MORADORES_IBGE']], on='CD_SETOR', how='left')

    # ==========================================
    # 6. Consumption Analysis & Behavioral Imputation
    # ==========================================
    print("Step 4: Merging 2015 consumption and applying behavioral imputation...")
    df_cons_2015 = pd.read_csv(FILE_CONS_2015, sep=';')
    df_final = pd.merge(df_enriched, df_cons_2015[['SK_MATRICULA', 'HCLQTCON']], on='SK_MATRICULA', how='left')
    
    df_final['HCLQTCON'] = df_final['HCLQTCON'].fillna(0)
    
    # Calculate Residents for Analysis (NN_MORADORES_ANALISE)
    # If original residents = 0, we use the IBGE sector average
    df_final['NN_MORADORES_ANALISE'] = df_final['NN_MORADORES'].copy()
    df_final.loc[df_final['NN_MORADORES'] == 0, 'NN_MORADORES_ANALISE'] = \
        df_final['NN_MEDIA_MORADORES_IBGE'].fillna(1).round(0)
    df_final.loc[df_final['NN_MORADORES_ANALISE'] < 1, 'NN_MORADORES_ANALISE'] = 1

    # Mine 2015 statistics from households with valid meter readings
    df_valid = df_final[df_final['HCLQTCON'] > 0].copy()
    df_valid['DAILY'] = (df_valid['HCLQTCON'] * 1000 / df_valid['NN_MORADORES_ANALISE']) / 30.5
    df_valid['PROFILE'] = df_valid['DAILY'].apply(classify_profile)
    averages_2015 = df_valid.groupby('PROFILE')['HCLQTCON'].mean().to_dict()

    # Impute consumption: If 0.0, use the Moderate profile average from 2015
    def impute_consumption(row):
        if row['HCLQTCON'] <= 0:
            return averages_2015.get('MODERADO', 15.0)
        return row['HCLQTCON']

    df_final['NN_MEDIA_CONSUMO'] = df_final.apply(impute_consumption, axis=1)
    df_final['NN_CONSUMO_DIARIO'] = (df_final['NN_MEDIA_CONSUMO'] * 1000 / df_final['NN_MORADORES_ANALISE']) / 30.5
    df_final['TP_COMPORTAMENTO'] = df_final['NN_CONSUMO_DIARIO'].apply(classify_profile)
        
    # Adjusted Income with PNAD Factor and fallback for missing sector data
    df_final['VL_RENDA_2015'] = (df_final['VL_RENDA_MEDIA_RESPONSAVEL'].fillna(df_final['VL_RENDA_MEDIA_RESPONSAVEL'].median())) * PNAD_FACTOR_2015
    
    # Sector simplification (Fixing slice for placeholder/long sectors)        
    df_final, sector_mapping = create_sector_mapping(
    df=df_final,
    sector_column='CD_SETOR_ORIGINAL',
    output_dir=BASE_PATH + '\\resultados',
    mapping_filename='de_para_setores_consumidores_2015.csv')

    # ==========================================
    # 7. Final Formatting and Export
    # ==========================================
    print("Step 5: Exporting final master baseline...")
    
    columns_order = [
        'SK_MATRICULA', 'NM_LOCALIDADE', 'NM_CATEGORIATARIFARIA', 'NM_SITUACAO_IMOVEL', 
        'NN_MORADORES', 'NN_MORADORES_ANALISE', 'ST_PISCINA', 'LAT_GEO', 'LONG_GEO', 
        'X', 'Y', 'CD_SETOR_ORIGINAL', 'CD_SETOR_SIMPLIFICADO', 'NN_MEDIA_CONSUMO', 
        'NN_CONSUMO_DIARIO', 'TP_COMPORTAMENTO', 'VL_RENDA_MEDIA_RESPONSAVEL', 
        'NN_MEDIA_MORADORES_IBGE', 'HCLQTCON', 'VL_RENDA_2015'
    ]
    
    df_export = df_final[columns_order].rename(columns={'CD_SETOR_SIMPLIFICADO': 'CD_SETOR'})
    df_export.to_csv(OUTPUT_FILE, index=False, sep=';')
    
    print("-" * 30)
    print(f"PROCESS COMPLETE: {len(df_export)} Agents Prepared.")
    print(f"Original Records: {original_count} | Exported: {len(df_export)}")
    print(f"File Saved: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

--- Starting Robust Baseline Generation (19k Master) ---
Step 1: Loading Master Table and converting to UTM X/Y...
Step 2: Performing Spatial Join with Census Tracts...


Step 3: Integrating IBGE Census Data...


C:\Users\Edmilson\AppData\Local\Temp\ipykernel_20384\4025515562.py:150: DtypeWarning: Columns (1,3,4,5,7,8,9,10,11,13,14,15,16,17,20,21,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,52,53,54,56,57,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,77,78,84,87) have mixed types. Specify dtype option on import or set low_memory=False.
  census_df = pd.read_csv(FILE_CENSUS, sep=';')


Step 4: Merging 2015 consumption and applying behavioral imputation...



--- Sector Mapping ---
Unique sectors mapped: 103
Format: S001 to S103
DE-PARA file saved at: ..\resultados\de_para_setores_consumidores_2015.csv
Total households mapped: 19630
Step 5: Exporting final master baseline...


------------------------------
PROCESS COMPLETE: 19630 Agents Prepared.
Original Records: 19630 | Exported: 19630
File Saved: ..\includes\Tabela_consumidores_Itapua_com_setor_comportamento_e_renda_2015.csv
